In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [7]:
BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
FIG_DIR = BASE_DIR / "figures" / "part3"
OUT_DIR = BASE_DIR / "outputs" / "part3"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
cpi_path = RAW_DIR / "CPI_DATA.xlsx"
cpi_path

PosixPath('/Users/marwanferreira/Desktop/audusd-carry-uip-ppp/data/raw/CPI_DATA.xlsx')

In [9]:
xls = pd.ExcelFile(cpi_path)
xls.sheet_names

['CPI- US DATA',
 'CPI - Canada',
 'CPI - Mexico',
 'CPI - Japan',
 'CPI - EURO',
 'CPI - Australia']

In [11]:
us_sheet = xls.sheet_names[0]
au_sheet = xls.sheet_names[5]

cpi_us_raw = pd.read_excel(cpi_path, sheet_name=us_sheet)
cpi_au_raw = pd.read_excel(cpi_path, sheet_name=au_sheet)

print("US shape:", cpi_us_raw.shape)
print("AU shape:", cpi_au_raw.shape)

cpi_us_raw.head(10)

US shape: (1358, 4)
AU shape: (456, 2)


,observation_date,Unnamed: 1,Period,Actual (Revised)
0,1913-01-01,1912-12-31,1912-12-31,9.8
1,1913-02-01,1913-01-31,1913-01-31,9.8
2,1913-03-01,1913-02-28,1913-02-28,9.8
3,1913-04-01,1913-03-31,1913-03-31,9.8
4,1913-05-01,1913-04-30,1913-04-30,9.7
5,1913-06-01,1913-05-31,1913-05-31,9.8
6,1913-07-01,1913-06-30,1913-06-30,9.9
7,1913-08-01,1913-07-31,1913-07-31,9.9
8,1913-09-01,1913-08-31,1913-08-31,10.0
9,1913-10-01,1913-09-30,1913-09-30,10.0


In [12]:
print("US columns:", cpi_us_raw.columns.tolist())
print("AU columns:", cpi_au_raw.columns.tolist())

US columns: ['observation_date', 'Unnamed: 1', 'Period', 'Actual (Revised)']
AU columns: ['Period', 'Actual (Revised)']


In [13]:
cpi_us = cpi_us_raw[["observation_date", "Actual (Revised)"]].copy()
cpi_us.columns = ["Date", "cpi_us"]

cpi_us["Date"] = pd.to_datetime(cpi_us["Date"])
cpi_us["cpi_us"] = pd.to_numeric(cpi_us["cpi_us"], errors="coerce")

cpi_us = cpi_us.dropna().sort_values("Date").reset_index(drop=True)
cpi_us.head()

,Date,cpi_us
0,1913-01-01,9.8
1,1913-02-01,9.8
2,1913-03-01,9.8
3,1913-04-01,9.8
4,1913-05-01,9.7


In [14]:
cpi_au = cpi_au_raw[["Period", "Actual (Revised)"]].copy()
cpi_au.columns = ["Date", "cpi_au"]

cpi_au["Date"] = pd.to_datetime(cpi_au["Date"])
cpi_au["cpi_au"] = pd.to_numeric(cpi_au["cpi_au"], errors="coerce")

cpi_au = cpi_au.dropna().sort_values("Date").reset_index(drop=True)
cpi_au.head()

,Date,cpi_au
0,1988-03-31,33.630000
1,1988-04-30,33.823333
2,1988-05-31,34.016667
3,1988-06-30,34.210000
4,1988-07-31,34.430000


In [23]:
cpi_us["Month"] = cpi_us["Date"].dt.to_period("M")
cpi_au["Month"] = cpi_au["Date"].dt.to_period("M")

cpi = pd.merge(
    cpi_au[["Month", "cpi_au"]],
    cpi_us[["Month", "cpi_us"]],
    on="Month",
    how="inner"
)

cpi["Date"] = cpi["Month"].dt.to_timestamp("M")
cpi = cpi[["Date", "cpi_au", "cpi_us"]].copy()

cpi["infl_au"] = np.log(cpi["cpi_au"]).diff()
cpi["infl_us"] = np.log(cpi["cpi_us"]).diff()
cpi["infl_diff"] = cpi["infl_au"] - cpi["infl_us"]

cpi = cpi[(cpi["Date"] >= "2009-01-01") & (cpi["Date"] <= "2023-12-31")].copy()
cpi.reset_index(drop=True, inplace=True)

print(cpi.shape)
cpi.head(12)

(180, 6)


,Date,cpi_au,cpi_us,infl_au,infl_us,infl_diff
0,2009-01-31,64.183333,211.143,0.000364,0.004343,-0.003979
1,2009-02-28,64.206667,212.193,0.000363,0.004961,-0.004597
2,2009-03-31,64.230000,212.709,0.000363,0.002429,-0.002065
3,2009-04-30,64.336667,213.240,0.001659,0.002493,-0.000834
4,2009-05-31,64.443333,213.856,0.001657,0.002885,-0.001228
5,2009-06-30,64.550000,215.693,0.001654,0.008553,-0.006899
6,2009-07-31,64.753333,215.351,0.003145,-0.001587,0.004732
7,2009-08-31,64.956667,215.834,0.003135,0.002240,0.000895
8,2009-09-30,65.160000,215.969,0.003125,0.000625,0.002500
9,2009-10-31,65.276667,216.177,0.001789,0.000963,0.000826


In [24]:
cpi.to_csv(PROCESSED_DIR / "audusd_cpi_processed.csv", index=False)